# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('annual-co2-emissions-per-country.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Entity'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 29384 rows | Countries: 247 | Years: 1750-2024
        Entity Code  Year  Annual CO₂ emissions
0  Afghanistan  AFG  1949               14656.0
1  Afghanistan  AFG  1950               84272.0
2  Afghanistan  AFG  1951               91600.0
3  Afghanistan  AFG  1952               91600.0
4  Afghanistan  AFG  1953              106256.0


In [6]:
# Explore before building

print("Countries:", df['Entity'].unique())
print("\nColumn names:", df.columns.tolist())
print("CO2 range:", df['Annual CO₂ emissions'].min(), "to", df['Annual CO₂ emissions'].max(), "Mt")
print("\nYears available:", df['Year'].min(), "to", df['Year'].max())
print("\nSample data:")
print(df.head(10))


Countries: 247

Column names: ['Entity', 'Code', 'Year', 'Annual CO₂ emissions']
CO2 range: 0.0 to 38598580000.0 Mt

Years available: 1750 to 2024

Sample data:
        Entity Code  Year  Annual CO₂ emissions
0  Afghanistan  AFG  1949               14656.0
1  Afghanistan  AFG  1950               84272.0
2  Afghanistan  AFG  1951               91600.0
3  Afghanistan  AFG  1952               91600.0
4  Afghanistan  AFG  1953              106256.0
5  Afghanistan  AFG  1954              106256.0
6  Afghanistan  AFG  1955              153888.0
7  Afghanistan  AFG  1956              183200.0
8  Afghanistan  AFG  1957              293120.0
9  Afghanistan  AFG  1958              329760.0


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [9]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
import plotly.graph_objects as go

# There is no 'region' or 'country' column in this dataset,
# so use Entity and Annual CO₂ emissions instead
asia_df = df.copy()

# Choose the country to highlight
highlight = "India"   # change this if you want

fig = go.Figure()

# 1. Add all countries in grey
for entity in asia_df['Entity'].unique():
    country_data = asia_df[asia_df['Entity'] == entity]
    
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['Annual CO₂ emissions'],
        mode='lines',
        line=dict(color='#DDDDDD', width=1),
        showlegend=False
    ))

# 2. Add highlighted country in red
highlight_data = asia_df[asia_df['Entity'] == highlight]

fig.add_trace(go.Scatter(
    x=highlight_data['Year'],
    y=highlight_data['Annual CO₂ emissions'],
    mode='lines+text',
    line=dict(color='red', width=3),
    text=[highlight] + [""] * (len(highlight_data) - 1),
    textposition='middle right',
    textfont=dict(color='red', size=14),
    showlegend=False
))

# 3. Layout
fig.update_layout(
    title=f"CO₂ Emissions Over Time — Highlight: {highlight}",
    xaxis_title="Year",
    yaxis_title="CO₂ Emissions (Mt)",
    template="plotly_white",
    height=500
)

fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [12]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# 1. Compute regional averages for 2000 and 2022
regional_avg = df.rename(columns={'Region':'region','Year':'year','Annual CO₂ emissions':'co2_mt','Entity':'entity'}).copy()

# If there's no region column, try to map countries to continents; fallback to entity (country) if mapping unavailable
if 'region' not in regional_avg.columns or regional_avg['region'].isnull().all():
    try:
        regional_avg['region'] = coco.convert(names=regional_avg['entity'], to='continent')
    except Exception:
        regional_avg['region'] = regional_avg['entity']

regional_avg = regional_avg.groupby(['region','year'])['co2_mt'].mean().reset_index()

# Filter only 2000 and 2022
slope = regional_avg[regional_avg['year'].isin([2000, 2022])]

# Pivot for easier comparison
pivot = slope.pivot(index='region', columns='year', values='co2_mt').reset_index()
pivot.columns = ['region', 'y2000', 'y2022']

# Determine color: increase = red, decrease = green
pivot['color'] = pivot.apply(lambda row: 'red' if row['y2022'] > row['y2000'] else 'green', axis=1)

fig = go.Figure()

# 2. Add one line per region
for _, row in pivot.iterrows():
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row['y2000'], row['y2022']],
        mode='lines+markers+text',
        line=dict(color=row['color'], width=3),
        text=[f"{row['region']} {row['y2000']:.1f}", f"{row['y2022']:.1f}"],
        textposition=['middle left', 'middle right'],
        showlegend=False
    ))

# 3. Layout formatting
fig.update_layout(
    title="Regional CO₂ Emissions Change (2000 → 2022)",
    xaxis=dict(title="", tickmode='array', tickvals=[2000, 2022]),
    yaxis=dict(title="", showticklabels=False),
    template="plotly_white",
    height=550
)

fig.show()
